# GMRES 调优实验

## 9.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info
command -v msprof


<a id="s-01-02_distributed_gmres_profiling"></a>


## 9.2 分布式 GMRES Profiling

### 本节学习目标

- 拆解 GMRES 阶段
- 理解 profile 字段与边界

### 先理解 GMRES 在求什么

我们要解 `Ax=b`。从初始解 x₀ 出发，残差是 `r₀=b−Ax₀`，它表示当前解还差多少。
GMRES 在由 `r₀, Ar₀, A²r₀, …` 张成的空间中寻找修正量，使 `||b−Ax||₂` 尽量小。

直接保存这些向量会越来越不稳定，因此 Arnoldi 每一步先做 SpMV，再把新向量在已有基向量上的分量减掉（Dot + AXPY），最后归一化（Norm + Scale）。
Host 用小型 Hessenberg 矩阵和 Givens 旋转解最小二乘问题，更新 x。

`restart=m` 表示最多保留 m 步基向量，然后用更新后的残差重新开始。m 小时用的内存少，但可能需要更多迭代；m 大时正交化和存储开销更高。
最后重新计算真实相对残差 `||b−Ax||₂/||b||₂`，以 `1e-6` 判断收敛。

### 一次正交化的小例子

已有单位方向 `v=(1,0)`，新向量 `w=(2,3)`。Dot 得到投影系数 `h=v·w=2`；AXPY 计算 `w−h×v=(0,3)`；Norm 得到长度 3；Scale 除以 3 得到新单位方向 `(0,1)`。

GMRES 对已有的每个方向重复投影和相减。restart 越大，最多保留的方向越多，每轮 Dot/AXPY 的工作也可能更多。


### 阶段拆解

每个 Arnoldi 步先做 SpMV，再用 Dot/AXPY 去掉已有方向上的分量，用 Norm/Scale 归一化。Host 处理小型 Hessenberg 矩阵和 Givens 旋转。

多卡时，每个 rank 先清零一个全长设备向量，只把局部数据写入自己的不重叠区间，再做 AllReduce SUM。例如 `[a,b,0,0]+[0,0,c,d]=[a,b,c,d]`，这样就能收齐 SpMV 所需的完整向量。Dot/Norm 的标量则另做求和归约；单卡跳过这些通信。


### Profiling 字段

工程记录 SpMV、Dot、AXPY、Norm、Givens、HCCL communication、ACL transfer、synchronization、AllReduce/AllGather calls。kernel launch 统计 Ascend C RTC kernel 的 Host 提交开销；正式路径中 SpMV/Dot/AXPY/Scale 均为真实 Device kernel，msprof 中应能看到对应的 kernel/task 时间。注意当前 Device 路径不使用 AllGather（全局向量经 Device placement + AllReduce 形成），其计数为 0。

### 时间的包含关系

```text
一次 solve 总时间
  ├─ RTC/设备状态准备、Host 控制等
  └─ SpMV、Dot、AXPY、Norm、Givens 等阶段
       └─ 各阶段内包含提交以及等待完成
```

Sync、Launch、Transfer 是同一过程的补充观察项，不再加到阶段时间上。多 rank 时，每轮等最慢进程完成，再对重复轮次求平均。


### 预期现象与结果分析

这些阶段是应用级测量。SpMV/Dot/AXPY 由 Ascend C RTC Device kernel 执行，其 Device 侧耗时应结合 msprof timeline 读取；HCCL 与 ACL transfer 是实际 Device 通信/搬移路径。

### 课后实践

将一次 Arnoldi 迭代映射到 profile 字段和 collective。

参考答案见 `answer/09.02_answer.md`。

### 建立单卡基线

下面构建并运行 Device GMRES。SpMV、Dot、AXPY 和 Scale 在设备端计算；Norm 复用设备 Dot 求平方和，再由 Host 开方。Host 还处理小型 Hessenberg 矩阵和 Givens 旋转。

首次运行自动生成矩阵，并调整为严格对角占优；缺少的对角元会补入，所以 nnz 可能比第 02 章的同名矩阵多，以本次输出为准。以后直接读取缓存。程序生成已知解 `x_true`，用 `b=A*x_true` 构造右端项，因此无需另传 b，也能计算解误差。

本实验设置 `warmup=0`。每次求解都会创建内部 RTC/Device 状态；矩阵读取、分区和通信组初始化在重复求解之前完成。记录求解时间时保留这一范围说明。

In [ ]:
%%bash
set -e
cd src/dis_gmres
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DDIS_GMRES_FORCE_STUB=OFF
cmake --build build -j4
# 学生修改：比较重启长度时修改 --restart，其他参数保持一致。
./build/bin/dis_gmres --matrix U2 --rank 0 --world-size 1 --device 0 \
  --warmup 0 --repeat 3 --restart 30 --max-iterations 300 --tolerance 1e-6


### 采集并导出 Profile

下面使用与基线相同的参数。`msprof` 会在 `PROFILE_DIR` 下创建本次采集的子目录；重新运行时可沿用这个设置，后面的代码读取最新一组结果。


In [ ]:
# 修改这里可更换 Profile 保存位置。
%env PROFILE_DIR=profiling/u2_r1_baseline


In [ ]:
%%bash
set -e
cd src/dis_gmres
msprof --application="./build/bin/dis_gmres --matrix U2 --rank 0 --world-size 1 --device 0 --warmup 0 --repeat 3 --restart 30 --max-iterations 300 --tolerance 1e-6" \
  --output="$PROFILE_DIR" --runtime-api=on --task-time=on


### 查看耗时最多的 API 和算子

CANN 9.0 导出的 `api_statistic_*.csv` 已统计各 API 的累计时间，`op_statistic_*.csv` 已统计算子的累计时间和占比。下面分别按耗时排序，显示前 10 项。

先看 API 表。Host 等待期间 Device 也可能在计算，因此两张表的时间不能直接相加；应用总时间仍看程序输出或时间线。

本节命令使用 `--warmup 0 --repeat 3`。程序的 total 和各阶段时间是三次求解的单次均值；msprof 表是本次采集期间的累计量，覆盖三次求解及采集范围内的准备工作。例如同步均值约 8443.54 ms，乘 3 约为 25.33 s，对应 API 表的 25328710.39 us。比较时先对齐次数和单位；API 与算子表还可能描述重叠的等待、计算区间，不能相加得到程序总时间。


CANN 9.0 的这条采集命令会自动导出结果。日志中的“no summary”针对相应的数据类别；本例后面能成功读到 API 和算子统计表，即可继续分析，完整日志可留在文件中。

`aclrtSynchronizeStream` 与内部 `StreamSynchronize` 可能描述同一次等待的不同调用层次，按调用层分别观察，不把两行相加成新的总时间。


In [ ]:
import csv
import os
from pathlib import Path

profile = Path("src/dis_gmres") / os.environ["PROFILE_DIR"]
# 同一次采集可能导出多次，读取最后一次导出的表。
api_file = sorted(profile.glob("PROF_*/mindstudio_profiler_output/api_statistic_*.csv"))[-1]
with api_file.open(encoding="utf-8-sig", newline="") as stream:
    api_rows = list(csv.DictReader(stream))
print(api_file)
for row in sorted(api_rows, key=lambda r: float(r["Time(us)"]), reverse=True)[:10]:
    print(row["API Name"], row["Time(us)"], "us", row["Count"], "次")


再看算子表。`Ratio(%)` 是该算子在本表累计时间中的占比。名称与应用阶段这样对应：

| 设备算子 | 应用阶段 |
|---|---|
| `gmres_spmv` | SpMV |
| `gmres_dot` | Dot，以及 Norm 的平方和计算 |
| `gmres_axpy`、`gmres_scale`、`gmres_sub` | 都归入 AXPY |

Norm 复用 Dot(v,v)，应用把这次调用的时间移到 Norm，平方根在 Host 求。因此算子表中没有单独的 Norm，不能只按名称逐项对齐。


In [ ]:
op_file = sorted(profile.glob("PROF_*/mindstudio_profiler_output/op_statistic_*.csv"))[-1]
with op_file.open(encoding="utf-8-sig", newline="") as stream:
    op_rows = list(csv.DictReader(stream))
print(op_file)
for row in sorted(op_rows, key=lambda r: float(r["Total Time(us)"]), reverse=True)[:10]:
    print(row["OP Type"], row["Total Time(us)"], "us", row["Ratio(%)"], "%")


<a id="s-01-03_baseline_and_bottleneck_analysis"></a>


## 9.3 Baseline 与瓶颈识别

### 本节学习目标

- 建立正确性 baseline
- 依据占比而非猜测选择瓶颈

### 检查求解结果

输出应为 `converged=yes`，相对残差不超过 `1e-6`、解误差小于 `1e-3`。


### Baseline 设计

固定矩阵、restart、tolerance、warmup、repeat、rank、线程预算和正交化方式。记录 total 及阶段计时，再按占比、调用次数和 rank 到达差异选择优化对象。

### 分析时间

先确认求解收敛，再比较总时间和各阶段占比。重复运行可观察耗时波动。


### 分析本次基线

先检查残差、解误差和退出码，再从本次 Profile 找出主要耗时。重复测量，观察总时间和各阶段是否稳定。

旧版 CPU 计算加 HCCL 的数据保存在 `src/dis_gmres/README.md`，可作为分析方法的参考；本节设备版本使用本次测量结果。

### 课后实践

选择一个矩阵，说明基线使用的参数、求解结果和主要耗时。

参考答案见 `answer/09.03_answer.md`。


<a id="s-01-04_compute_and_communication_tuning"></a>


## 9.4 计算与通信调优

### 本节学习目标

- 理解 MGS 正交化与重启长度的影响
- 控制 collective 与线程预算

### 正交化与重启长度

MGS 依次减去已有基向量的投影，通常每个投影都需要标量规约。CGS 可以把多个系数合并规约，但当前 Device 代码未实现 CGS，因此本实验使用 MGS。

单卡先比较 restart=4 与 30，观察 Dot/AXPY 时间、实际迭代数与总时间。重启长度同时影响每轮正交化成本和收敛过程，不能只看单次 SpMV 快慢。

### 计算后端与线程数

正式路径的 SpMV/Dot/AXPY/Scale 由 Ascend C RTC kernel 执行。`--no-openmp` 与 `--unfused-vector-ops` 只用于 Host 实现，不作为本节 Device 调优变量。多 rank 的 CPU 线程预算用于控制 Host 资源占用，不能据此推断 Device 计算加速。

### 结果分析

比较 restart=4 与 30 的实际迭代数、Dot/AXPY 时间和总时间。较大的搜索空间可能改善收敛，但每轮正交化和存储开销也会增加。
本次 U2 实测中，restart=4 用 13 步、约 3733 ms，restart=30 用 36 步、约 10826 ms。这里要区分理想的 GMRES 性质和当前 FP32 实现的数值表现。

`src/dis_gmres/src/npu_gmres.cpp` 每步只做一遍 MGS，没有再正交化。内层使用 `abs(g[j+1])/||b||` 估计残差；更新 x 后重新计算真实残差 `||b-Ax||/||b||`，未达阈值就继续下一轮。

补充诊断只增加残差记录和基向量回读，未改变求解计算。把第一轮基向量回读到 Host，用 double 计算不同基向量的归一化内积绝对值，取其中最大值；理想正交时应接近 0。

| restart | 第一轮最大归一化内积 | 最终迭代数 | 最终真实相对残差 |
|---|---|---|---|
| 4 | 0.0149 | 13 | 5.36e-7 |
| 30 | 0.6357 | 36 | 5.18e-7 |

长基向量已明显失去正交性，搜索空间变大并未带来理想的残差最小化效果。restart=30 到第 30 步的真实残差仍为 `4.33e-4`，重启后到第 36 步才达标；restart=4 在第 12 步为 `1.68e-6`，第 13 步达标。两种结果都经过真实残差检查，本次较短重启更有效。诊断运行增加了观察开销，耗时比较仍使用前面的原始测量。


### 课后实践

固定矩阵、MGS、容差、最大迭代数和重复次数，只改变 restart。列出两次的残差、解误差、迭代数与时间，并解释差异。

参考答案见 `answer/09.04_answer.md`。

<a id="s-01-05_memory_scheduling_and_scaling"></a>


## 9.5 内存、调度与扩展性

### 本节学习目标

- 分析 buffer reuse 和 Host/Device 搬移与同步
- 比较 partition 与 rank scaling

### 内存与同步

一次 solve 内，CSR 和 Krylov 向量常驻设备，缓冲区在迭代间复用。但每次 Kernel 提交后仍同步等待，Dot/Norm 还要把标量回读到 Host，多卡时另有 HCCL 同步。本次单卡的 HCCL 时间为 0，大量同步来自 Kernel 完成等待；这段时间也包含设备计算，不能全部当作同步调用本身的开销。


### 分区与多卡扩展（选读）

rows 按行数分配，nnz 按非零元数量尽量均衡。均匀矩阵上两者可能接近；长尾矩阵上按行数分配容易让少数 rank 承担更多计算。单 rank 时两种分区都拿到完整矩阵，不能用来做性能对照。

有多卡时，固定矩阵和求解参数比较分区，记录各 rank 工作量、通信和总时间；单卡课程完成上一节的 restart 比较即可。

### 分析结果

结合总时间判断优化效果，再看各阶段变化，解释计算、传输和同步的开销。


### 课后实践

比较 rows/nnz 分区或不同 rank 数，写出需要同时控制的变量。

参考答案见 `answer/09.05_answer.md`。